# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/begs44/flyrank-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: ranking (scored with a binary-classification proxy).** The team can review about 50 pages a week out of 30,000, and over 16,000 pages are labelled declining. A yes/no label for every page does not help; the editor needs an **order**: who goes first. So a model outputs a score per page, and pages are reviewed from the top down.

In [11]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/begs44/flyrank-ml/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# Ranking, not yes/no: capacity is tiny compared to the number of pages
CAPACITY = 50
print(f"Pages: {len(df):,} | weekly review capacity: {CAPACITY} pages ({CAPACITY / len(df):.2%} of all pages)")
print(f"Pages labelled 'declining': {(df['trend_direction'] == 'down').sum():,} -> far more than the team can review,")
print("so the output must be an ORDER (who first), not just a yes/no label.")

Pages: 30,000 | weekly review capacity: 50 pages (0.17% of all pages)
Pages labelled 'declining': 16,262 -> far more than the team can review,
so the output must be an ORDER (who first), not just a yes/no label.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target proxy:** `target_decay` = 1 when a page's impressions in the last 30 days were more than 20% below the previous 30 days (`trend_direction == "down"`). It is an **observed outcome** from search data, not a product decision. It stands in for "worth a refresh review", because true lost value cannot be measured before a review. The cell checks the rule: every label-1 page has `trend_pct` ≤ −20%.

In [12]:
# Target proxy: 1 = impressions fell >20% (last 30 days vs previous 30 days)
df["target_decay"] = (df["trend_direction"] == "down").astype(int)
base = df["target_decay"].mean()
print(f"target_decay = 1 for {df['target_decay'].sum():,} of {len(df):,} pages | base rate {base:.3f}")
print("\nHow the label is built (checked on the raw windows):")
check = df.loc[df["impressions_prev_30d"] > 0, ["trend_pct", "target_decay"]]
print(check.groupby("target_decay")["trend_pct"].agg(["min", "max", "count"]).round(1).to_string())

target_decay = 1 for 16,262 of 30,000 pages | base rate 0.542

How the label is built (checked on the raw windows):
                min      max  count
target_decay                       
0             -20.0  44900.0  10350
1            -100.0    -20.0  16262


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision@50**, the share of truly declining pages among the 50 highest-ranked, which matches the weekly batch size. It is always reported **next to the base rate (0.542)**: a random order already scores about 0.54, and by luck alone it lands anywhere between roughly 0.42 and 0.66. **"Good" = P@50 ≥ 0.70** (35 of 50), clearly above what luck gives.

In [13]:
def precision_at_k(y_true, scores, k=50):
    """Share of truly declining pages among the k highest-scored pages."""
    top = np.argsort(-np.asarray(scores), kind="stable")[:k]
    return np.asarray(y_true)[top].mean()

# What a RANDOM order scores: average P@50 over 1,000 shuffles = the bar to beat
rng = np.random.default_rng(42)
random_p50 = [precision_at_k(df["target_decay"], rng.random(len(df))) for _ in range(1000)]
print(f"Random order: P@50 = {np.mean(random_p50):.3f} on average (5th-95th percentile {np.percentile(random_p50, 5):.2f}-{np.percentile(random_p50, 95):.2f})")
print(f"Base rate:    {base:.3f}")
print(f"My 'good' bar: P@50 >= 0.70 (35 of 50 truly declining) - clearly above what luck gives")

Random order: P@50 = 0.540 on average (5th-95th percentile 0.42-0.66)
Base rate:    0.542
My 'good' bar: P@50 >= 0.70 (35 of 50 truly declining) - clearly above what luck gives


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one page** (`content_id`, unique). The dataframe below shows the pre-decision signals I plan to use (visibility, clicks, CTR, position, age, update history) next to the target. `avg_position = 0` means "no data", so it is set to missing, not treated as rank 0. `client_id` is kept only for grouping and splitting, never as an input.

In [14]:
# One row = one page. Pre-decision signals + the target, as a real dataframe
df["avg_position"] = df["avg_position"].replace(0, np.nan)   # 0 means "no data", not rank 0
cols = ["content_id", "client_id", "content_type", "impressions_90d", "clicks_90d", "ctr", "avg_position",
        "content_age_days", "days_since_last_update", "target_decay"]
print(f"one row = one page: {df['content_id'].is_unique} | {len(df):,} rows")
df[cols].head(8)

one row = one page: True | 30,000 rows


,content_id,client_id,content_type,impressions_90d,clicks_90d,ctr,avg_position,content_age_days,days_since_last_update,target_decay
0,content_304f48230142,client_f369cb89fc,keyword article,3803,29,0.76,10.6,187,20,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,7,0.05,20.3,445,25,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,11,0.09,36.5,141,20,1
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,58,0.49,6.2,463,22,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,24,0.13,44.0,263,14,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,3970,1,0.03,8.5,147,20,1
6,content_9a34b442b552,client_8722616204,keyword article,20,0,0.00,7.0,90,20,1
7,content_a63219c6e95a,client_19581e27de,keyword article,1724,1,0.06,21.2,445,22,0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

* **The usual fixed rules fail here.** "Old page", "not updated in 180+ days" and "age > 180 and position > 15" all pick pages that decline **less** often than the base rate (0.43–0.47 vs 0.54). They would make the batch worse than random.
* **Signals interact.** The decline rate depends on position *and* visibility together: pages in positions 1–3 with fewer than 100 impressions decline 26% of the time, but with 100–999 impressions 88% of the time. One threshold on one column cannot capture that.
* **So:** a learned score that combines several signals, tested on clients it never saw, and compared against a transparent rule baseline on the same metric. If it cannot beat the rule, the rule wins.

In [15]:
# Fixed rules people use - how do they do against the base rate?
rules = {
    "old page (age > 365 days)": df["content_age_days"] > 365,
    "not updated in 180+ days": df["days_since_last_update"] > 180,
    "age > 180 AND position > 15": (df["content_age_days"] > 180) & (df["avg_position"] > 15),
}
print(f"Base rate {base:.3f}")
for name, mask in rules.items():
    print(f"- {name:<30} flags {mask.sum():>6,} pages | share declining {df.loc[mask, 'target_decay'].mean():.3f}")

# Signals interact: decline rate by position band x visibility band (n in brackets)
df["position_band"] = pd.cut(df["avg_position"], [0, 3, 10, 20, 200], labels=["1-3", "4-10", "11-20", "20+"])
df["visibility"] = pd.cut(df["impressions_90d"], [-1, 99, 999, np.inf], labels=["<100", "100-999", "1000+"])
rate = df.pivot_table(index="position_band", columns="visibility", values="target_decay", aggfunc="mean", observed=True)
n = df.pivot_table(index="position_band", columns="visibility", values="target_decay", aggfunc="size", observed=True)
print("\nDecline rate by position x visibility (n):")
print((rate.round(2).astype(str) + " (" + n.astype(str) + ")").to_string())

Base rate 0.542
- old page (age > 365 days)      flags  6,360 pages | share declining 0.426
- not updated in 180+ days       flags    174 pages | share declining 0.471
- age > 180 AND position > 15    flags  7,491 pages | share declining 0.472

Decline rate by position x visibility (n):
visibility            <100      100-999        1000+
position_band                                       
1-3             0.26 (586)   0.88 (139)   0.71 (416)
4-10           0.47 (3182)  0.72 (2485)  0.56 (6175)
11-20          0.54 (1397)  0.65 (2456)  0.61 (3420)
20+            0.43 (1623)  0.47 (3414)  0.63 (3501)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.